# Bài 5a (Trang 45) - Online Retail (Germany)
## Câu 10: Tạo luật kết hợp với min_conf = 50% và in ra các luật

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
%matplotlib inline

df = pd.read_excel("../Dataset/Online Retail.xlsx")

### Các bước xử lý của câu 1 → 9 (chuẩn bị dữ liệu)

In [3]:
df["Description"] = df["Description"].str.strip()
df = df.dropna(axis=0, subset=["InvoiceNo"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

df = df[~df["InvoiceNo"].str.contains("C")]

grouped = (df[df["Country"] == "Germany"]
           .groupby(["InvoiceNo", "Description"])["Quantity"]
           .sum())

basket = grouped.unstack().reset_index().fillna(0).set_index("InvoiceNo")

def encode_units(x):
    return 1 if x >= 1 else 0

# Tương đương basket.map(encode_units) nhưng vector hóa cho nhanh; kiểu bool theo khuyến nghị của mlxtend
basket_sets = basket >= 1

basket_sets = basket_sets.drop(columns=["POSTAGE"], errors="ignore")

start = time.time()
frequent_itemsets = apriori(basket_sets, min_support=0.05, use_colnames=True)
apriori_time = time.time() - start

### Câu 10: Tạo luật kết hợp với min_conf = 50% và in ra các luật

In [4]:
rules = association_rules(frequent_itemsets, num_itemsets=len(basket_sets),
                          metric="confidence", min_threshold=0.5)

In [5]:
rules.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   antecedents         8 non-null      object 
 1   consequents         8 non-null      object 
 2   antecedent support  8 non-null      float64
 3   consequent support  8 non-null      float64
 4   support             8 non-null      float64
 5   confidence          8 non-null      float64
 6   lift                8 non-null      float64
 7   representativity    8 non-null      float64
 8   leverage            8 non-null      float64
 9   conviction          8 non-null      float64
 10  zhangs_metric       8 non-null      float64
 11  jaccard             8 non-null      float64
 12  certainty           8 non-null      float64
 13  kulczynski          8 non-null      float64
dtypes: float64(12), object(2)
memory usage: 1.0+ KB


In [6]:
rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))

In [7]:
print("Số luật tìm được:", len(rules))
for i, r in rules.iterrows():
    print(f"Rule {i + 1}: {r['antecedents']} => {r['consequents']} "
          f"(support = {r['support']:.3f}, confidence = {r['confidence']:.3f}, lift = {r['lift']:.3f})")

Số luật tìm được: 8
Rule 1: PLASTERS IN TIN CIRCUS PARADE => PLASTERS IN TIN WOODLAND ANIMALS (support = 0.068, confidence = 0.585, lift = 4.243)
Rule 2: PLASTERS IN TIN SPACEBOY => PLASTERS IN TIN WOODLAND ANIMALS (support = 0.061, confidence = 0.571, lift = 4.145)
Rule 3: PLASTERS IN TIN WOODLAND ANIMALS => ROUND SNACK BOXES SET OF4 WOODLAND (support = 0.074, confidence = 0.540, lift = 2.202)
Rule 4: RED RETROSPOT CHARLOTTE BAG => WOODLAND CHARLOTTE BAG (support = 0.059, confidence = 0.844, lift = 6.648)
Rule 5: ROUND SNACK BOXES SET OF 4 FRUITS => ROUND SNACK BOXES SET OF4 WOODLAND (support = 0.131, confidence = 0.833, lift = 3.400)
Rule 6: ROUND SNACK BOXES SET OF4 WOODLAND => ROUND SNACK BOXES SET OF 4 FRUITS (support = 0.131, confidence = 0.536, lift = 3.400)
Rule 7: SPACEBOY LUNCH BOX => ROUND SNACK BOXES SET OF4 WOODLAND (support = 0.070, confidence = 0.681, lift = 2.778)
Rule 8: WOODLAND CHARLOTTE BAG => ROUND SNACK BOXES SET OF4 WOODLAND (support = 0.063, confidence = 0.500, 

### Nhận xét
- Với min_conf = 50% thu được 8 luật, ví dụ: ROUND SNACK BOXES SET OF 4 FRUITS => ROUND SNACK BOXES SET OF4 WOODLAND (conf ≈ 0.83), RED RETROSPOT CHARLOTTE BAG => WOODLAND CHARLOTTE BAG (conf ≈ 0.84).